# Example 11.4 / 4: Diagnose Derivative Failures

Companion notebook for **11.4 Pitfalls of Nonlinear Programming** in the AMPL lab Documentation, section "Diagnose Derivative Failures". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Diagnose Derivative Failures

For ordinary AMPL expressions, AMPL supplies first and second derivative information to nonlinear
solvers through automatic differentiation. This avoids hand-coding gradients, Jacobians, and
Lagrangian Hessians, but it does not make an undefined or nonsmooth expression differentiable.
Imported functions must separately provide the evaluations and derivative support required by the
selected solver.

When a solver reports a derivative or evaluation problem, first identify the objective or
constraint and the point at which it failed. Check function domains, active bounds, branch points,
and scaling before changing solver tolerances. If the solver offers a derivative checker, compare
the supplied derivatives with finite differences at several domain-safe interior points. A
finite-difference disagreement is diagnostic evidence, not a reason to replace the modeled
function silently. Near singularities and kinks, finite differences can also be unreliable, so
interpret the comparison together with the mathematical domain.

A derivative check can be done by hand with AMPL parameters. At $x=6$, $L=10$ the analytic
derivative of $x/(1-x/L)$ is $1/(1-x/L)^2=6.25$; a central difference agrees, while at the kink
of `abs` the one-sided differences disagree:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param L := 10;    # capacity in x / (1 - x / L)
param x0 := 6;    # point where the derivative is checked
param h := 1e-4;  # finite-difference step

param f {t in {x0 - h, x0 + h}} = t / (1 - t / L);  # function values at x0 - h and x0 + h
param analytic = 1 / (1 - x0 / L)^2;                # exact derivative at x0
param central = (f[x0 + h] - f[x0 - h]) / (2 * h);  # central-difference estimate
param abs_forward = (abs(0 + h) - abs(0)) / h;      # forward difference of abs at its kink
param abs_backward = (abs(0) - abs(0 - h)) / h;     # backward difference of abs at its kink

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for name in ("analytic", "central", "abs_forward", "abs_backward"):
    print(name, round(ampl.param[name].value(), 6))

analytic 6.25
central 6.25
abs_forward 1
abs_backward -1
